<a href="https://colab.research.google.com/github/nirmalpate/ADVANCED-PDF-RAG-MODEL/blob/main/advanced_pdf_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced PDF RAG (Hybrid Search + Reranker)
Switch to a GPU first: Runtime -> Change runtime type -> T4 GPU.
Pipeline: PDF -> chunks -> embeddings -> Chroma + BM25 -> hybrid retrieval -> reranker -> LLM -> answer + sources

## 1. Install

In [ ]:
!pip install -q langchain-community langchain-chroma langchain-huggingface langchain-text-splitters langchain-classic pypdf sentence-transformers chromadb transformers accelerate rank_bm25

## 2. Imports

In [ ]:
import os
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("GPU available:", torch.cuda.is_available())

## 3. PDF folder (upload your PDFs into `pdf_data`)

In [ ]:
PDF_DIR = "./pdf_data"
os.makedirs(PDF_DIR, exist_ok=True)
print("Upload PDFs to:", os.path.abspath(PDF_DIR))

## 4. Load and chunk PDFs

In [ ]:
def load_and_chunk(directory, chunk_size=500, chunk_overlap=50):
    pdf_files = [f for f in os.listdir(directory) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print("No PDFs found. Upload to pdf_data and re-run.")
        return []

    documents = []
    for name in pdf_files:
        pages = PyPDFLoader(os.path.join(directory, name)).load()
        documents.extend(pages)
        print(f"Loaded {name}: {len(pages)} pages")

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

chunks = load_and_chunk(PDF_DIR)
assert chunks, "No chunks created. Upload a PDF first."

## 5. Embeddings and vector database

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
print("Vector database ready")

## 6. Advanced retriever: hybrid search + reranker

In [ ]:
# Keyword search (good for exact names, IDs, numbers)
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 5

# Semantic search (good for meaning)
vector = vectorstore.as_retriever(search_kwargs={"k": 5})

# Combine both
hybrid = EnsembleRetriever(retrievers=[bm25, vector], weights=[0.4, 0.6])

# Rerank combined results and keep the best 3
reranker = CrossEncoderReranker(
    model=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base"),
    top_n=3,
)
retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid,
)
print("Retriever ready")

## 7. Load the LLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # or TinyLlama/TinyLlama-1.1B-Chat-v1.0 (weaker)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    temperature=None,
    return_full_text=False,
)

local_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator))
print("LLM ready")

## 8. Build the RAG chain

In [ ]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you do not know the answer, say that you "
    "don't know. Keep the answer concise.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(local_llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
print("RAG chain ready")

## 9. Ask questions

In [ ]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print("Answer:", result["answer"].strip())
    print("\nSources:")
    for d in result["context"]:
        src = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page")
        page = page + 1 if isinstance(page, int) else "?"
        print(f"- {src} | page {page}")

ask("What is this document about?")